# 09. 후보지 스냅 — 실행 가능한 배치안 (보류)

## 이 노트북이 하는 일
MCLP가 고른 격자중심(08)을 실제 **24시간 접근 지점**(편의점·파출소·주차장·정류장)으로 옮겨 현실적 배치안을 만든다.

## 왜 이렇게 설계했나 (설계 이유)
- **왜 실좌표로 스냅하나:** 격자중심은 추상적 위치. AED는 실제 '관리주체·전원·야간개방'이 있는 지점에 놓여야 한다.
- **왜 24시간 시설만 1차 대상인가:** 심야 심정지 대응이 목적. 주간만 여는 곳은 야간 무용.
- **왜 반경 90m인가:** 격자(100m) 안에 들어오는 근접 시설만 인정. 너무 멀면 그 격자를 못 덮는다.
- **왜 결과가 아니라 '문제점 정리'로 보류하나:** (1)OSM이 한국 경로당·소규모 편의점을 대량 누락 → 후보지가 실제보다 적게 잡힘. (2)24h 시설도 저지대 편중이라 고위험 상단엔 매칭이 거의 없음. → 실배치안 확정은 부산시 공공시설 데이터 + 현장조사가 선행돼야 함.
- **핵심 발견(2차 논거):** 필요한 곳(산복도로 상단)엔 AED뿐 아니라 AED를 걸 24시간 시설조차 없다.

## 데이터 출처
- 후보지: OpenStreetMap(편의점·파출소·주차장·정류장 등). MCLP 위치: 08.

In [ ]:
import os, warnings                       # 폴더·경고
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd            # 수치·표
import geopandas as gpd, osmnx as ox        # 지리표·OSM 시설 조회
from scipy.spatial import cKDTree           # 최근접 시설 탐색
from shapely.geometry import box            # 조회 사각형
from shapely.ops import unary_union         # 경계 합치기
import folium                              # 배치 지도
CRS_WGS, CRS_M = 4326, 5186                 # 위경도 / 평면
R_SNAP = 90                                 # 스냅 허용 반경(m) — 격자(100m) 내 근접
os.makedirs("outputs", exist_ok=True)

## 1. MCLP 결과 + 대상지 경계

In [ ]:
mclp = pd.read_csv("outputs/mclp_new_aed.csv")                                        # 08의 신규 AED 후보(격자중심)
mclp_g = gpd.GeoDataFrame(mclp, geometry=gpd.points_from_xy(mclp["lon"],mclp["lat"]),
                          crs=CRS_WGS).to_crs(CRS_M)                                  # 점 → 5186
print("MCLP 신규 후보:", len(mclp_g))

adm = ox.features_from_polygon(box(129.020,35.100,129.075,35.155), tags={"boundary":"administrative"})  # 행정경계
adm = adm[adm.geometry.geom_type.isin(["Polygon","MultiPolygon"])]; adm["name"]=adm["name"].astype(str)
bnd = unary_union(adm[adm["name"].str.contains("초량|좌천",na=False)].geometry)         # 초량·좌천 경계
bnd_buf = gpd.GeoSeries([bnd],crs=CRS_WGS).to_crs(CRS_M).buffer(150).to_crs(CRS_WGS).iloc[0]  # 경계+150m(시설 검색영역)

## 2. 실제 후보지 수집 (OSM)

In [ ]:
TAGS = {"shop":"convenience", "amenity":["police","community_centre","parking","social_facility"],  # 찾을 시설 태그들
        "highway":"bus_stop"}
feat = ox.features_from_polygon(bnd_buf, tags=TAGS)                                    # 검색영역 내 시설 조회
feat = feat[feat.geometry.notna()].copy()                                            # 좌표 없는 것 제거
feat["geometry"] = feat.geometry.centroid                                            # 면이면 중심점으로

def categorize(r):                                                                    # 태그 → (유형, 24시간여부)
    if str(r.get("shop"))=="convenience": return ("편의점", True)                     # 편의점: 24h
    a = str(r.get("amenity"))
    if a=="police": return ("파출소/치안센터", True)                                   # 파출소: 24h
    if a=="parking": return ("공영주차장", True)                                       # 주차장: 24h
    if a=="community_centre": return ("주민센터/경로당", False)                         # 주민센터/경로당: 주간
    if a=="social_facility": return ("복지시설", False)                                # 복지시설: 주간
    if str(r.get("highway"))=="bus_stop": return ("버스정류장", True)                  # 정류장: 옥외 24h
    return ("기타", False)

cats = feat.apply(categorize, axis=1)                                                 # 각 시설 분류
feat["category"] = [c[0] for c in cats]                                               # 유형
feat["access24"] = [c[1] for c in cats]                                               # 24시간 여부
feat["poi_name"] = feat["name"].astype(str).replace("nan","(무명)") if "name" in feat.columns else "(무명)"  # 이름
poi = gpd.GeoDataFrame(feat[["poi_name","category","access24","geometry"]], crs=CRS_WGS).to_crs(CRS_M)  # 정리·5186
print("수집 후보지:", len(poi))
print(poi["category"].value_counts().to_string())
print("24시간 접근 후보:", int(poi["access24"].sum()))

## 3. 각 MCLP 위치 → 최근접 24시간 지점 스냅

In [ ]:
poi24 = poi[poi["access24"]].reset_index(drop=True)                                   # 24시간 시설만
p24_xy = np.c_[poi24.geometry.x.values, poi24.geometry.y.values]                       # 좌표 배열
tree = cKDTree(p24_xy)                                                                # KD트리
m_xy = np.c_[mclp_g.geometry.x.values, mclp_g.geometry.y.values]                       # MCLP 위치
dist, idx = tree.query(m_xy, k=1)                                                     # 각 MCLP의 최근접 24h 시설·거리

rows=[]                                                                               # 배치 제안 행 모음
for i,(_,r) in enumerate(mclp_g.iterrows()):                                          # MCLP 위치마다
    if dist[i] <= R_SNAP:                                                             # 반경 내 24h 시설이 있으면
        p = poi24.iloc[idx[i]]                                                        # 그 시설
        rows.append({"rank":int(r["rank"]), "dong":r["dong"], "risk":round(r["risk_norm"],3),
                     "제안유형":"기존시설 활용", "후보지":p["poi_name"], "유형":p["category"],
                     "거리m":round(dist[i]), "lon":poi24.to_crs(4326).iloc[idx[i]].geometry.x,
                     "lat":poi24.to_crs(4326).iloc[idx[i]].geometry.y})
    else:                                                                            # 없으면 신규 함체 필요
        rows.append({"rank":int(r["rank"]), "dong":r["dong"], "risk":round(r["risk_norm"],3),
                     "제안유형":"신규 설치함체 필요", "후보지":"(반경내 24h 지점 없음)", "유형":"-",
                     "거리m":round(dist[i]),
                     "lon":mclp.iloc[i]["lon"], "lat":mclp.iloc[i]["lat"]})
prop = pd.DataFrame(rows).sort_values("rank")                                          # 순위대로 정렬
n_reuse = (prop["제안유형"]=="기존시설 활용").sum()                                     # 기존시설 활용 가능 개수
print(f"기존시설 활용 가능: {n_reuse}/{len(prop)} | 신규 함체 필요: {len(prop)-n_reuse}")
print()
print(prop[["rank","dong","risk","제안유형","후보지","유형","거리m"]].to_string(index=False))

## 4. 저장 + 지도

In [ ]:
prop.to_csv("outputs/aed_siting_proposal.csv", index=False, encoding="utf-8-sig")     # 배치 제안 저장

m = folium.Map(location=[35.122,129.045], zoom_start=15, tiles="cartodbpositron")
for _,p in poi24.to_crs(CRS_WGS).iterrows():                                          # 모든 24h 후보지(연회색)
    folium.CircleMarker([p.geometry.y,p.geometry.x], radius=2, color="#bbb", fill=True,
                        tooltip=f'{p["category"]} {p["poi_name"]}').add_to(m)
for _,r in prop.iterrows():                                                           # 제안 지점
    color = "green" if r["제안유형"]=="기존시설 활용" else "orange"                     # 초록=기존활용 / 주황=신규함체
    icon = "plus" if color=="green" else "wrench"
    folium.Marker([r["lat"],r["lon"]],
        tooltip=f'#{int(r["rank"])} {r["dong"]} | {r["제안유형"]} | {r["후보지"]}({r["유형"]})',
        icon=folium.Icon(color=color, icon=icon, prefix="fa")).add_to(m)
legend=('<div style="position:fixed;bottom:30px;left:30px;z-index:9999;background:white;padding:8px 12px;'
        'border:1px solid #999;font-size:12px"><b>AED 배치 제안</b><br>'
        '<span style="color:green">●</span> 기존시설 활용(24h) &nbsp; '
        '<span style="color:orange">●</span> 신규 설치함체 &nbsp; '
        '<span style="color:#bbb">●</span> 24h 후보지</div>')
m.get_root().html.add_child(folium.Element(legend))
m.save("outputs/siting_map.html")                                                      # 배치 지도 저장
print("저장: outputs/aed_siting_proposal.csv, outputs/siting_map.html")
m